In [18]:
"""
Benchmark: direct inverse vs. Cholesky-solve for Fisher matrix
──────────────────────────────────────────────────────────────
• Covariance  : 50 independent  (3×7)×(3×7) blocks  → 1050×1050
• Jacobian    : random (1050 × P)   with P = 10 by default
• Backend     : JAX 0.4+  (GPU/CPU/TPU as available)
"""

import time
import jax
import jax.numpy as jnp
import jax.random as jr
import jax.scipy as jsp

# ─── Dimensions ──────────────────────────────────────────────────────────────
N_BLOCKS   = 100
N_CHAN     = 3
N_TIME     = 7
BLOCK_SIZE = N_CHAN * N_TIME     # 21
DIM        = N_BLOCKS * BLOCK_SIZE
P          = 82                  # number of model parameters

# ─── Generate a synthetic block-diagonal SPD covariance ──────────────────────
key = jr.PRNGKey(0)

def random_spd(k):
    m = jr.normal(k, (BLOCK_SIZE, BLOCK_SIZE))
    return (m @ m.T) + 1e-3 * jnp.eye(BLOCK_SIZE)  # regularise

C_blocks = jax.vmap(random_spd)(jr.split(key, N_BLOCKS))
C = jsp.linalg.block_diag(*C_blocks)                      # (DIM, DIM)

# ─── Random Jacobian (could come from autodiff in practice) ──────────────────
jac = jr.normal(key, (DIM, P))

# ─── Two Fisher implementations ──────────────────────────────────────────────
@jax.jit
def fisher_inv(cov, jac):
    cov_inv = jnp.linalg.inv(cov)
    return jac.T @ cov_inv @ jac

@jax.jit
def fisher_chol(cov, jac):
    lufac = jsp.linalg.lu_factor(cov)
    Cinv_jac = jsp.linalg.lu_solve(lufac,jac)
    return jac.T @ Cinv_jac

# ─── Compile & time ──────────────────────────────────────────────────────────
def time_fn(fn, *args, n_repeat=10):
    t0 = time.perf_counter()
    out = fn(*args)
    jax.block_until_ready(out)          # compile + first execution
    compile_first = time.perf_counter() - t0

    t0 = time.perf_counter()
    for _ in range(n_repeat):
        out = fn(*args)
    jax.block_until_ready(out)
    avg_exec = (time.perf_counter() - t0) / n_repeat
    return compile_first, avg_exec, out

c_inv_first, c_inv_exec, F_inv  = time_fn(fisher_inv,  C, jac)
c_ch_first,  c_ch_exec,  F_ch   = time_fn(fisher_chol, C, jac)

# ─── Numerical cross-check ───────────────────────────────────────────────────
rel_err = jnp.linalg.norm(F_inv - F_ch) / jnp.linalg.norm(F_inv)

# ─── Report ──────────────────────────────────────────────────────────────────
print(f"Matrix dimension            : {DIM} × {DIM}")
print(f"Jacobian columns (P)         : {P}")
print("── Timing (ms) ──────────────────────────────")
print(f" direct inverse  – compile+first : {c_inv_first*1e3:8.2f}  |  avg exec : {c_inv_exec*1e3:8.2f}")
print(f" Cholesky-solve – compile+first : {c_ch_first *1e3:8.2f}  |  avg exec : {c_ch_exec *1e3:8.2f}")
print("── Accuracy ────────────────────────────────")
print(f"‖F_inv − F_ch‖₂ / ‖F_inv‖₂  = {rel_err:.3e}")

Matrix dimension            : 2100 × 2100
Jacobian columns (P)         : 82
── Timing (ms) ──────────────────────────────
 direct inverse  – compile+first :   123.04  |  avg exec :    79.26
 Cholesky-solve – compile+first :    80.32  |  avg exec :    39.64
── Accuracy ────────────────────────────────
‖F_inv − F_ch‖₂ / ‖F_inv‖₂  = 3.332e-07
